In [ ]:
# set up: desktop
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import sqlite3
import time
import seaborn as sns

# path set up:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

from src.config import DATA_RAW, DATA_PROCESSED
from src import eda

# database connection set up:
DB = '/Users/admin/Desktop/carbon-portfolio-project-v2/data/carbon.db'
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")

root on path: /Users/admin/Desktop/carbon-portfolio-project-v2


In [1]:
# set up: Google Colab
import sys
import sqlite3
import time
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:

# mount drive (data artifacts live here — never in git)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# clone fresh, or pull if it already exists (so re-running the cell doesn't error)
import os
REPO = "/content/repo"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/salinela/carbon-portfolio-project-v2.git {REPO}
%cd {REPO}

Cloning into '/content/repo'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 149 (delta 81), reused 102 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 296.26 KiB | 7.05 MiB/s, done.
Resolving deltas: 100% (81/81), done.
/content/repo


In [4]:
# root on path — mirrors desktop's package-style imports
sys.path.insert(0, REPO)
print("root on path:", REPO)

root on path: /content/repo


In [18]:
# live-reload src edits after a git pull without restarting the runtime
from src import eda
from src import feature_engineering as fe
from src.config import DATA_RAW, DATA_PROCESSED

In [9]:
# DB copied to LOCAL disk (not the Drive FUSE mount) to avoid SQLite locking.
# Needed to read/query it, not just to rebuild — copy once per session.
DRIVE = "/content/drive/MyDrive/carbon_project_v2"
if not os.path.exists("/content/carbon.db"):
    !cp "{DRIVE}/carbon.db" /content/carbon.db

In [10]:
DB = "/content/carbon.db"
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")
print("fe MIN_PERIODS_FRAC:", fe.MIN_PERIODS_FRAC, "| batched:", hasattr(fe, "build_price_features_batched"))

fe MIN_PERIODS_FRAC: 0.7 | batched: True


one-time usage

In [ ]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# one-time cleanup of the stale broken view
con.execute("DROP VIEW IF EXISTS v_company_emissions;")
con.commit()

# Phase 0: Data Assembly

In [11]:
meta, meta_diag = eda.build_meta(con)
fy,   fy_diag   = eda.build_firm_year(con)

meta_diag, fy_diag

({'n_companies': 8288,
  'by_universe': {'EU': 7988, 'ETS': 300},
  'eligible_n': 5883,
  'sector_nulls_master': 58,
  'sector_nulls_after_backfill': 58,
  'country_nulls': 0,
  'bvd_nulls': 0,
  'coverage_status_counts': {'mapped_loaded': 5883,
   'unmapped_exchange': 1773,
   'mapped_no_data': 523,
   'no_ticker': 109}},
 {'n_rows': 12487,
  'n_companies': 1306,
  'year_range': (2012, 2025),
  'by_source': {'trucost': 9214, 'ets_registry': 3273},
  'revenue_nulls': 573,
  'intensity_nulls': 573})

In [12]:
fy_ids = set(fy["company_id"])
elig   = meta[meta["eligible"] == 1]
mask   = elig.index.isin(fy_ids)
print("eligible:", len(elig))
print("eligible w/ emissions:", int(mask.sum()))
print(elig[mask]["universe"].value_counts().to_dict())

eligible: 5883
eligible w/ emissions: 1259
{'EU': 978, 'ETS': 281}


## Phase 1: Universe Characterization

### Section A: Compute carbon itensity tiers (source registry x 1-digit NACE x year)

In [13]:
# --- cohort flags on meta (enables optional carbon-blind comparison) ---
fy_ids = set(fy["company_id"])
meta["has_emissions_data"] = meta.index.isin(fy_ids).astype(int)
meta["carbon_sample"] = ((meta["eligible"] == 1) &
                         (meta["has_emissions_data"] == 1)).astype(int)

# cross-check against master's has_emissions flag
print("has_emissions agree:",
      (meta["has_emissions"] == meta["has_emissions_data"]).mean())
print("carbon_sample n:", int(meta["carbon_sample"].sum()))

has_emissions agree: 0.9639237451737451
carbon_sample n: 1259


In [14]:
# --- nace1 for tiering: master, backfilled from orbis, first digit ---
nace_full = meta["nace_code"].fillna(meta["orbis_nace_code"])
nace1 = nace_full.astype("string").str.extract(r"(\d)")[0]   # index = company_id

# --- recompute tiers ---
fy, tier_diag = eda.compute_tiers(fy, nace1)
tier_diag

{'tier_counts': {'non_ets_high': 2947,
  'non_ets_low': 2911,
  'non_ets_medium': 2872,
  'ets_high': 1074,
  'ets_low': 1044,
  'ets_medium': 1015,
  <NA>: 573,
  'ets_untiered': 41,
  'non_ets_untiered': 10},
 'nace1_nulls': 0,
 'n_untiered': 51,
 'n_tiered': 11863}

In [15]:
d = meta[meta["has_emissions"] != meta["has_emissions_data"]]
print(len(d))
print(d.groupby(["has_emissions", "has_emissions_data"]).size())
print(d["universe"].value_counts().to_dict())

299
has_emissions  has_emissions_data
0              1                     299
dtype: int64
{'ETS': 299}


### Section B: Monthly Tier-based Portfolio Returns Helper

Main objective: for each month, take every firm currently sitting per tier and average their forward returns (equal-weighted basket)

tier_portfolio_returns produces one such series per tie; "do high-carbon baskets earn different returns than low-carbon ones"

The attach_tier_asof step is what tells each company-month which basket it was in at that date, using the 1-July lag so you're never using an emissions figure before it was public.

In [21]:
# panel load — desktop: your processed dir; Colab: DRIVE

features = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/features_month_end.parquet")
label    = pd.read_parquet("/content/drive/MyDrive/carbon_project_v2/label_fwd_return.parquet")

In [26]:
panel

amihud  beta_252   beta_63  boll_bw_20  boll_bw_60  \
company_id   date                                                               
API          2013-01-31  0.076450       NaN       NaN    0.021481         NaN   
             2013-02-28  0.043239       NaN       NaN    0.022293    0.054812   
             2013-03-29  0.037164       NaN  0.305769    0.050888    0.054112   
             2013-04-30  0.027683       NaN  0.318868    0.039862    0.053324   
             2013-05-31  0.022100       NaN  0.173202    0.038202    0.133239   
...                           ...       ...       ...         ...         ...   
VGG6564A1057 2026-02-27  0.000785  0.101318 -0.030778    0.180008    0.148224   
             2026-03-31  0.000964  0.076216  0.158639    0.109952    0.440837   
             2026-04-30  0.001230  0.190725  0.174548    0.051969    0.497835   
             2026-05-29  0.001314  0.387715  0.569576    0.170408    0.120299   
             2026-06-29  0.001447  0.306588  0.541468    0.157881    0.167884   

                         boll_pctb_20  boll_pctb_60  brent_beta_252  \
company_id   date                                                     
API          2013-01-31     -0.060404           NaN             NaN   
             2013-02-28      0.528651      0.273635             NaN   
             2013-03-29           NaN           NaN             NaN   
             2013-04-30      0.515689      0.689152             NaN   
             2013-05-31      0.766510      0.230156             NaN   
...                               ...           ...             ...   
VGG6564A1057 2026-02-27     -0.231960     -0.262970        0.013616   
             2026-03-31      0.231356      0.135498       -0.063146   
             2026-04-30      0.577518      0.336190       -0.070962   
             2026-05-29      0.653145      0.794202       -0.083390   
             2026-06-29      1.079233      1.253537       -0.094656   

                         brent_beta_63  dollar_vol  ...  stoch_d_63  \
company_id   date                                   ...               
API          2013-01-31            NaN   11.367876  ...         NaN   
             2013-02-28            NaN   11.712449  ...         NaN   
             2013-03-29       0.167161   12.284985  ...         NaN   
             2013-04-30       0.248134   12.225194  ...   72.222222   
             2013-05-31       0.041055   12.614666  ...   13.888889   
...                                ...         ...  ...         ...   
VGG6564A1057 2026-02-27       0.143344   16.729047  ...   29.832132   
             2026-03-31      -0.044426   16.519517  ...    6.961816   
             2026-04-30      -0.065166   15.920974  ...    6.666654   
             2026-05-29      -0.097330   16.641054  ...   54.863814   
             2026-06-29      -0.114282   16.215601  ...   92.566907   

                         stoch_k_14  stoch_k_63  trend_dist_sma200  \
company_id   date                                                    
API          2013-01-31   25.000000         NaN                NaN   
             2013-02-28   50.000000         NaN                NaN   
             2013-03-29         NaN         NaN                NaN   
             2013-04-30   60.000000   66.666667                NaN   
             2013-05-31   71.090027   25.000000                NaN   
...                             ...         ...                ...   
VGG6564A1057 2026-02-27   12.629412   12.629412          -0.219732   
             2026-03-31   16.071372    4.931490          -0.275586   
             2026-04-30   72.151951    7.945205          -0.220685   
             2026-05-29   74.289427   58.524198          -0.139133   
             2026-06-29   95.138846   96.929797          -0.009879   

                         trend_sma50_200  us10y_beta_252  us10y_beta_63  \
company_id   date                                                         
API          2013-01-31              NaN             NaN        

In [25]:
# pivot to wide:
panel = features.pivot_table(index=["company_id", "date"],
                             columns="signal_name", values="value")

panel = panel.join(label.set_index(["company_id", "date"])["fwd_ret"])
print("panel:", panel.shape)

panel: (730161, 42)


In [27]:

panel_t = eda.attach_tier_asof(panel, fy)
print("tier coverage:", round(panel_t["carbon_tier"].notna().mean(), 3))

tier coverage: 0.184


In [28]:
tret = eda.tier_portfolio_returns(panel_t)
print(tret.shape)
tret.tail()

(155, 8)


carbon_tier,ets_high,ets_low,ets_medium,ets_untiered,non_ets_high,non_ets_low,non_ets_medium,non_ets_untiered
date,,,,,,,,
2026-01-30,NaN,NaN,NaN,NaN,0.001464,0.020621,-0.019596,NaN
2026-02-27,NaN,NaN,NaN,NaN,-0.057135,-0.056221,-0.082630,NaN
2026-03-31,NaN,NaN,NaN,NaN,0.044266,0.047350,0.072799,NaN
2026-04-30,NaN,NaN,NaN,NaN,0.043705,0.013508,0.048464,NaN
2026-05-29,NaN,NaN,NaN,NaN,-0.027886,-0.015544,-0.011606,NaN


In [39]:
# 1) firms actually CONTRIBUTING per tier per year (drives tret NaN directly)
chk = (panel_t.reset_index()[["company_id", "date", "carbon_tier", "fwd_ret"]]
       .dropna(subset=["carbon_tier", "fwd_ret"]))

# remove untiered:
chk = chk[~chk["carbon_tier"].astype(str).str.endswith("untiered")]

# see number of companies in each yearly tier:
chk["year"] = pd.to_datetime(chk["date"]).dt.year
chk.groupby(["year", "carbon_tier"])["company_id"].nunique().unstack("carbon_tier")

carbon_tier,ets_high,ets_low,ets_medium,non_ets_high,non_ets_low,non_ets_medium
year,,,,,,
2013,68.0,75.0,70.0,NaN,NaN,NaN
2014,81.0,87.0,92.0,144.0,148.0,140.0
2015,83.0,81.0,81.0,182.0,176.0,182.0
2016,87.0,86.0,83.0,183.0,177.0,189.0
2017,86.0,86.0,79.0,273.0,268.0,285.0
2018,91.0,86.0,83.0,304.0,307.0,316.0
2019,94.0,88.0,89.0,312.0,312.0,326.0
2020,95.0,91.0,93.0,327.0,334.0,353.0
2021,94.0,85.0,87.0,347.0,364.0,380.0


In [40]:
# 2) is the bottleneck upstream (emissions/revenue) or the price panel?
fyt = fy[fy["carbon_tier"].notna()
         & ~fy["carbon_tier"].astype(str).str.endswith("untiered")]

# number of companies per year in the yearly dataframe:
fyt.groupby(["year", "source"]).size().unstack("source")

source,ets_registry,trucost
year,,
2012,246.0,NaN
2013,260.0,446.0
2014,266.0,472.0
2015,267.0,489.0
2016,272.0,769.0
2017,274.0,825.0
2018,275.0,846.0
2019,273.0,913.0
2020,272.0,936.0


In [77]:
tret_trim = tret.loc["2014-07-30":"2025-06-30"].drop(columns=['ets_untiered', 'non_ets_untiered'])
tret_trim.isna().mean()

,0
carbon_tier,
ets_high,0.015152
ets_low,0.030303
ets_medium,0.030303
non_ets_high,0.000000
non_ets_low,0.015152
non_ets_medium,0.015152


In [78]:
tret_trim[tret_trim.isna().any(axis = 1)]

carbon_tier,ets_high,ets_low,ets_medium,non_ets_high,non_ets_low,non_ets_medium
date,,,,,,
2018-02-28,-0.035744,NaN,NaN,-0.031747,-0.014578,0.003955
2018-03-30,0.000761,NaN,NaN,-0.028747,0.063794,0.038382
2024-02-29,NaN,NaN,NaN,0.084667,NaN,NaN
2024-03-29,NaN,NaN,NaN,-0.023028,NaN,NaN
